In [ ]:
import networkx as nx
import random
from typing import Set, Tuple, List
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Fig 2 (a) only for the legends : high critical R, high seperator K

# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]) -> Tuple[Set[int], Set[int], Set[int]]:
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1: direct neighbors of K ONLY)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    # r is in R_connected iff r has at least one direct neighbor in K_final
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Critical shielded R inside R_connected
# Conditions:
#   (1) fragile: (deg-1)/deg_init < threshold
#   (2) has >= 2 fragile neighbors counted within R_final (R-only)
# ---------------------------
def get_critical_shielded_R(G: nx.Graph, R_final: Set[int], R_connected: Set[int], threshold: float) -> Set[int]:
    # Baseline (pre-cascade) degrees at evaluation time
    init_deg = {n: G.degree(n) for n in R_final}
    # Fragile set across ALL R
    fragile_R = {r for r in R_final if init_deg[r] > 0 and (G.degree(r) - 1) / init_deg[r] < threshold}

    critical = set()
    for r in R_connected:
        if r not in fragile_R:
            continue
        # Count fragile neighbors restricted to R_final (R-only)
        fragile_neighbors_in_R = sum(1 for nbr in G.neighbors(r) if (nbr in fragile_R) and (nbr in R_final))
        if fragile_neighbors_in_R >= 2:
            critical.add(r)
    return critical

# ---------------------------
# Graph coloring helper (color whole graph; avg degree per color)
# ---------------------------
def avg_color_degree(H: nx.Graph):
    # You can choose strategy="largest_first" if you want; default is fine too
    colors_dict = nx.coloring.greedy_color(H)
    if not colors_dict:
        return {}, {}
    max_color = max(colors_dict.values())
    avg_color_dict = {}
    for c in range(max_color + 1):
        nodes_c = [n for n, col in colors_dict.items() if col == c]
        if nodes_c:
            degrees = np.array([H.degree(n) for n in nodes_c], dtype=float)
            avg_color_dict[c] = float(np.mean(degrees))
        else:
            avg_color_dict[c] = 0.0
    return colors_dict, avg_color_dict

def get_coloring_critical_subset(G: nx.Graph, critical_R_original: Set[int]) -> Set[int]:
    # Color whole graph, compute lowest average degree among color classes
    _, avg_color_dict = avg_color_degree(G)
    if not avg_color_dict:
        return set()
    lowest_avg_deg = min(avg_color_dict.values())
    # Keep only those critical R with degree > lowest average degree
    return {r for r in critical_R_original if G.degree(r) > lowest_avg_deg}

# ---------------------------
# Cascade (shielded set never fails)
# - Seeds in R_isolated
# - Propagates in R_isolated and R_connected_nonshielded
# - Stops at shielded R; K/T never fail
# - Remove-on-failure
# ---------------------------
def run_r_isolated_to_r_connected_cascade(
    G_orig: nx.Graph,
    R_isolated: Set[int],
    R_connected: Set[int],
    shielded_R: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
) -> Tuple[Set[int], Set[int]]:
    """
    Returns (failed_R_isolated, failed_R_connected_nonshielded).
    """
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}

    R_connected_nonshielded = R_connected - shielded_R

    failed_R_iso: Set[int] = set()
    failed_R_con: Set[int] = set()

    # Seed failures only from R_isolated
    candidates = list(R_isolated)
    if not candidates:
        return failed_R_iso, failed_R_con
    n0 = min(n_initial_failures, len(candidates))
    start_nodes = random.sample(candidates, n0)
    failed_R_iso.update(start_nodes)
    for node in start_nodes:
        if node in G:
            G.remove_node(node)

    # Cascade loop
    while True:
        new_fail_iso: Set[int] = set()
        new_fail_con: Set[int] = set()

        # 1) R_isolated updates
        for node in sorted(R_isolated - failed_R_iso):
            if node in shielded_R:
                continue
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_iso.add(node)

        # 2) R_connected (non-shielded only) updates
        for node in sorted(R_connected_nonshielded - failed_R_con):
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_con.add(node)

        if not new_fail_iso and not new_fail_con:
            break

        for nf in (new_fail_iso | new_fail_con):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con

    return failed_R_iso, failed_R_con

# ---------------------------
# Main simulation
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 2

    surviving_fraction_T = []
    critical_R_fraction = []           # original 2-condition critical R / total R
    critical_R_fraction_coloring = []  # coloring-refined critical R / total R

    for threshold in threshold_values:
        T_frac_runs = []
        critR_frac_runs = []
        critR_color_frac_runs = []

        # per-threshold summary
        Rconn_counts = []
        critR_counts = []
        critR_color_counts = []
        subset_ok_all_runs = True          # original critical ⊆ R_connected (by construction)
        subset_color_ok_all_runs = True    # coloring-critical ⊆ original critical

        for seed in range(n_runs):
            random.seed(seed)
            G = nx.barabasi_albert_graph(nodenumber, 3, seed=seed)

            # Select top nodes as target (T)
            top_n = max(1, int(nodenumber * target_pct))
            degree_list = sorted(G.degree, key=lambda x: x[1], reverse=True)
            target_nodes = [node for node, _ in degree_list[:top_n]]

            # Initial T/K/R
            T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)

            # TIA refinement
            T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

            # --- C1 split AFTER TIA ---
            R_connected = get_R_connected_direct(G, R_final, K_final)
            R_isolated = R_final - R_connected

            # 1) Original critical R (2 conditions) on current graph
            critical_R = get_critical_shielded_R(G, R_final, R_connected, threshold)

            # 2) Graph-coloring refinement: keep only critical R with degree > lowest avg color degree
            critical_R_coloring = get_coloring_critical_subset(G, critical_R)

            # Collect counts
            Rconn_counts.append(len(R_connected))
            critR_counts.append(len(critical_R))
            critR_color_counts.append(len(critical_R_coloring))
            subset_ok_all_runs &= critical_R.issubset(R_connected)
            subset_color_ok_all_runs &= critical_R_coloring.issubset(critical_R)

            # 3) Cascade: shield ONLY the coloring-based critical R
            _failed_R_iso, _failed_R_con_nonshielded = run_r_isolated_to_r_connected_cascade(
                G_orig=G,
                R_isolated=R_isolated,
                R_connected=R_connected,
                shielded_R=critical_R_coloring,
                threshold=threshold,
                n_initial_failures=n_initial_failures,
            )

            # Metrics

            failed_R_total = _failed_R_iso | _failed_R_con_nonshielded

            failed_K = set()
            failed_T = set()

            init_deg = dict(G.degree())

            # ---- K failure ----
            for k in K_final:
                if any(n in failed_R_total for n in G.neighbors(k)):
                    if init_deg[k] > 0 and (G.degree(k) / init_deg[k]) < threshold:
                       failed_K.add(k)

            # ---- T failure ----
            for t in T_final:
                if any(n in failed_K for n in G.neighbors(t)):
                    if init_deg[t] > 0 and (G.degree(t) / init_deg[t]) < threshold:
                       failed_T.add(t)

            # ---- Survival fraction ----
            survival = 1 - len(failed_T) / max(len(T_final), 1)
            T_frac_runs.append(survival)

            # ---- Critical R fractions ----
            denom = max(len(R_final), 1)
            critR_frac_runs.append(len(critical_R) / denom)
            critR_color_frac_runs.append(len(critical_R_coloring) / denom)

        surviving_fraction_T.append(np.mean(T_frac_runs))
        critical_R_fraction.append(np.mean(critR_frac_runs))
        critical_R_fraction_coloring.append(np.mean(critR_color_frac_runs))

        print(
            f"Threshold={threshold:.2f} | "
            f"Avg Surviving T={np.mean(T_frac_runs):.3f} | "
            f"Avg R_connected(C1)={np.mean(Rconn_counts):.1f} | "
            f"Avg Critical_R(orig)={np.mean(critR_counts):.1f} | "
            f"Avg Critical_R(color)={np.mean(critR_color_counts):.1f} | "
            f"Avg Critical R frac (orig)={np.mean(critR_frac_runs):.3f} | "
            f"Avg Critical R frac (color)={np.mean(critR_color_frac_runs):.3f} | "
            f"subset_ok(orig⊆R_con)={subset_ok_all_runs} | "
            f"subset_ok(color⊆orig)={subset_color_ok_all_runs}"
        )

    # ---------------------------
    # Plot: Surviving T vs Threshold
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, surviving_fraction_T, marker='o', label='Surviving T Fraction')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction")
    plt.title(f"Surviving T Fraction vs Threshold (shielded = coloring-critical; {n_runs} runs)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

    # Combined Plot: Critical R fractions (Original vs Graph Coloring)
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, critical_R_fraction, marker='s', label='Critical R Fraction (Original)')
    plt.plot(threshold_values, critical_R_fraction_coloring, marker='^', label='Critical R Fraction (Graph Coloring)')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction of Total R")
    plt.title("Critical R fraction vs Threshold\nOriginal vs Graph Coloring Based")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()


In [ ]:
# Fig 2 (a) only for the legends : low critical R, low separator K


# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]) -> Tuple[Set[int], Set[int], Set[int]]:
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1: direct neighbors of K ONLY)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    # r is in R_connected iff r has at least one direct neighbor in K_final
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Critical shielded R inside R_connected
# Conditions:
#   (1) fragile: (deg-1)/deg_init < threshold
#   (2) has >= 2 fragile neighbors counted within R_final (R-only)
# ---------------------------
def get_critical_shielded_R(G: nx.Graph, R_final: Set[int], R_connected: Set[int], threshold: float) -> Set[int]:
    # Baseline (pre-cascade) degrees at evaluation time
    init_deg = {n: G.degree(n) for n in R_final}
    # Fragile set across ALL R
    fragile_R = {r for r in R_final if init_deg[r] > 0 and (G.degree(r) - 1) / init_deg[r] < threshold}

    critical = set()
    for r in R_connected:
        if r not in fragile_R:
            continue
        # Count fragile neighbors restricted to R_final (R-only)
        fragile_neighbors_in_R = sum(1 for nbr in G.neighbors(r) if (nbr in fragile_R) and (nbr in R_final))
        if fragile_neighbors_in_R >= 2:
            critical.add(r)
    return critical

# ---------------------------
# Graph coloring helper (color whole graph; avg degree per color)
# ---------------------------
def avg_color_degree(H: nx.Graph):
    # You can choose strategy="largest_first" if you want; default is fine too
    colors_dict = nx.coloring.greedy_color(H)
    if not colors_dict:
        return {}, {}
    max_color = max(colors_dict.values())
    avg_color_dict = {}
    for c in range(max_color + 1):
        nodes_c = [n for n, col in colors_dict.items() if col == c]
        if nodes_c:
            degrees = np.array([H.degree(n) for n in nodes_c], dtype=float)
            avg_color_dict[c] = float(np.mean(degrees))
        else:
            avg_color_dict[c] = 0.0
    return colors_dict, avg_color_dict

def get_coloring_critical_subset(G: nx.Graph, critical_R_original: Set[int]) -> Set[int]:
    # Color whole graph, compute lowest average degree among color classes
    _, avg_color_dict = avg_color_degree(G)
    if not avg_color_dict:
        return set()
    lowest_avg_deg = min(avg_color_dict.values())
    # Keep only those critical R with degree > lowest average degree
    return {r for r in critical_R_original if G.degree(r) > lowest_avg_deg}

# ---------------------------
# Cascade (shielded set never fails)
# - Seeds in R_isolated
# - Propagates in R_isolated and R_connected_nonshielded
# - Stops at shielded R; K/T never fail
# - Remove-on-failure
# ---------------------------
def run_r_isolated_to_r_connected_cascade(
    G_orig: nx.Graph,
    R_isolated: Set[int],
    R_connected: Set[int],
    shielded_R: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
) -> Tuple[Set[int], Set[int]]:
    """
    Returns (failed_R_isolated, failed_R_connected_nonshielded).
    """
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}

    R_connected_nonshielded = R_connected - shielded_R

    failed_R_iso: Set[int] = set()
    failed_R_con: Set[int] = set()

    # Seed failures only from R_isolated
    candidates = list(R_isolated)
    if not candidates:
        return failed_R_iso, failed_R_con
    n0 = min(n_initial_failures, len(candidates))
    start_nodes = random.sample(candidates, n0)
    failed_R_iso.update(start_nodes)
    for node in start_nodes:
        if node in G:
            G.remove_node(node)

    # Cascade loop
    while True:
        new_fail_iso: Set[int] = set()
        new_fail_con: Set[int] = set()

        # 1) R_isolated updates
        for node in sorted(R_isolated - failed_R_iso):
            if node in shielded_R:
                continue
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_iso.add(node)

        # 2) R_connected (non-shielded only) updates
        for node in sorted(R_connected_nonshielded - failed_R_con):
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_con.add(node)

        if not new_fail_iso and not new_fail_con:
            break

        for nf in (new_fail_iso | new_fail_con):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con

    return failed_R_iso, failed_R_con

# ---------------------------
# Main simulation
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 2

    surviving_fraction_T = []
    critical_R_fraction = []           # original 2-condition critical R / total R
    critical_R_fraction_coloring = []  # coloring-refined critical R / total R

    for threshold in threshold_values:
        T_frac_runs = []
        critR_frac_runs = []
        critR_color_frac_runs = []

        # per-threshold summary
        Rconn_counts = []
        critR_counts = []
        critR_color_counts = []
        subset_ok_all_runs = True          # original critical ⊆ R_connected (by construction)
        subset_color_ok_all_runs = True    # coloring-critical ⊆ original critical

        for seed in range(n_runs):
            random.seed(seed)
            G = nx.barabasi_albert_graph(nodenumber, 3, seed=seed)

            # Select bottom-degree nodes as target (T)
            top_n = max(1, int(nodenumber * target_pct))
            degree_list = sorted(G.degree, key=lambda x: x[1])  # lowest degree first
            target_nodes = [node for node, _ in degree_list[:top_n]]

            # Initial T/K/R
            T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)

            # TIA refinement
            T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

            # --- C1 split AFTER TIA ---
            R_connected = get_R_connected_direct(G, R_final, K_final)
            R_isolated = R_final - R_connected

            # 1) Original critical R (2 conditions) on current graph
            critical_R = get_critical_shielded_R(G, R_final, R_connected, threshold)

            # 2) Graph-coloring refinement: keep only critical R with degree > lowest avg color degree
            critical_R_coloring = get_coloring_critical_subset(G, critical_R)

            # Collect counts
            Rconn_counts.append(len(R_connected))
            critR_counts.append(len(critical_R))
            critR_color_counts.append(len(critical_R_coloring))
            subset_ok_all_runs &= critical_R.issubset(R_connected)
            subset_color_ok_all_runs &= critical_R_coloring.issubset(critical_R)

            # 3) Cascade: shield ONLY the coloring-based critical R
            _failed_R_iso, _failed_R_con_nonshielded = run_r_isolated_to_r_connected_cascade(
                G_orig=G,
                R_isolated=R_isolated,
                R_connected=R_connected,
                shielded_R=critical_R_coloring,
                threshold=threshold,
                n_initial_failures=n_initial_failures,
            )

            # Metrics

            failed_R_total = _failed_R_iso | _failed_R_con_nonshielded

            failed_K = set()
            failed_T = set()

            init_deg = dict(G.degree())

            # ---- K failure ----
            for k in K_final:
                if any(n in failed_R_total for n in G.neighbors(k)):
                    if init_deg[k] > 0 and (G.degree(k) / init_deg[k]) < threshold:
                       failed_K.add(k)

            # ---- T failure ----
            for t in T_final:
                if any(n in failed_K for n in G.neighbors(t)):
                    if init_deg[t] > 0 and (G.degree(t) / init_deg[t]) < threshold:
                       failed_T.add(t)

            # ---- Survival fraction ----
            survival = 1 - len(failed_T) / max(len(T_final), 1)
            T_frac_runs.append(survival)

            # ---- Critical R fractions ----
            denom = max(len(R_final), 1)
            critR_frac_runs.append(len(critical_R) / denom)
            critR_color_frac_runs.append(len(critical_R_coloring) / denom)

        surviving_fraction_T.append(np.mean(T_frac_runs))
        critical_R_fraction.append(np.mean(critR_frac_runs))
        critical_R_fraction_coloring.append(np.mean(critR_color_frac_runs))

        print(
            f"Threshold={threshold:.2f} | "
            f"Avg Surviving T={np.mean(T_frac_runs):.3f} | "
            f"Avg R_connected(C1)={np.mean(Rconn_counts):.1f} | "
            f"Avg Critical_R(orig)={np.mean(critR_counts):.1f} | "
            f"Avg Critical_R(color)={np.mean(critR_color_counts):.1f} | "
            f"Avg Critical R frac (orig)={np.mean(critR_frac_runs):.3f} | "
            f"Avg Critical R frac (color)={np.mean(critR_color_frac_runs):.3f} | "
            f"subset_ok(orig⊆R_con)={subset_ok_all_runs} | "
            f"subset_ok(color⊆orig)={subset_color_ok_all_runs}"
        )

    # ---------------------------
    # Plot: Surviving T vs Threshold
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, surviving_fraction_T, marker='o', label='Surviving T Fraction')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction")
    plt.title(f"Surviving T Fraction vs Threshold (shielded = coloring-critical; {n_runs} runs)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

    # Combined Plot: Critical R fractions (Original vs Graph Coloring)
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, critical_R_fraction, marker='s', label='Critical R Fraction (Original)')
    plt.plot(threshold_values, critical_R_fraction_coloring, marker='^', label='Critical R Fraction (Graph Coloring)')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction of Total R")
    plt.title("Critical R fraction vs Threshold\nOriginal vs Graph Coloring Based")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()


In [ ]:
# Fig 2 (a) only for the legends : random critical R, random separator K


# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]) -> Tuple[Set[int], Set[int], Set[int]]:
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1: direct neighbors of K ONLY)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    # r is in R_connected iff r has at least one direct neighbor in K_final
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Critical shielded R inside R_connected
# Conditions:
#   (1) fragile: (deg-1)/deg_init < threshold
#   (2) has >= 2 fragile neighbors counted within R_final (R-only)
# ---------------------------
def get_critical_shielded_R(G: nx.Graph, R_final: Set[int], R_connected: Set[int], threshold: float) -> Set[int]:
    # Baseline (pre-cascade) degrees at evaluation time
    init_deg = {n: G.degree(n) for n in R_final}
    # Fragile set across ALL R
    fragile_R = {r for r in R_final if init_deg[r] > 0 and (G.degree(r) - 1) / init_deg[r] < threshold}

    critical = set()
    for r in R_connected:
        if r not in fragile_R:
            continue
        # Count fragile neighbors restricted to R_final (R-only)
        fragile_neighbors_in_R = sum(1 for nbr in G.neighbors(r) if (nbr in fragile_R) and (nbr in R_final))
        if fragile_neighbors_in_R >= 2:
            critical.add(r)
    return critical

# ---------------------------
# Graph coloring helper (color whole graph; avg degree per color)
# ---------------------------
def avg_color_degree(H: nx.Graph):
    # You can choose strategy="largest_first" if you want; default is fine too
    colors_dict = nx.coloring.greedy_color(H)
    if not colors_dict:
        return {}, {}
    max_color = max(colors_dict.values())
    avg_color_dict = {}
    for c in range(max_color + 1):
        nodes_c = [n for n, col in colors_dict.items() if col == c]
        if nodes_c:
            degrees = np.array([H.degree(n) for n in nodes_c], dtype=float)
            avg_color_dict[c] = float(np.mean(degrees))
        else:
            avg_color_dict[c] = 0.0
    return colors_dict, avg_color_dict

def get_coloring_critical_subset(G: nx.Graph, critical_R_original: Set[int]) -> Set[int]:
    # Color whole graph, compute lowest average degree among color classes
    _, avg_color_dict = avg_color_degree(G)
    if not avg_color_dict:
        return set()
    lowest_avg_deg = min(avg_color_dict.values())
    # Keep only those critical R with degree > lowest average degree
    return {r for r in critical_R_original if G.degree(r) > lowest_avg_deg}

# ---------------------------
# Cascade (shielded set never fails)
# - Seeds in R_isolated
# - Propagates in R_isolated and R_connected_nonshielded
# - Stops at shielded R; K/T never fail
# - Remove-on-failure
# ---------------------------
def run_r_isolated_to_r_connected_cascade(
    G_orig: nx.Graph,
    R_isolated: Set[int],
    R_connected: Set[int],
    shielded_R: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
) -> Tuple[Set[int], Set[int]]:
    """
    Returns (failed_R_isolated, failed_R_connected_nonshielded).
    """
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}

    R_connected_nonshielded = R_connected - shielded_R

    failed_R_iso: Set[int] = set()
    failed_R_con: Set[int] = set()

    # Seed failures only from R_isolated
    candidates = list(R_isolated)
    if not candidates:
        return failed_R_iso, failed_R_con
    n0 = min(n_initial_failures, len(candidates))
    start_nodes = random.sample(candidates, n0)
    failed_R_iso.update(start_nodes)
    for node in start_nodes:
        if node in G:
            G.remove_node(node)

    # Cascade loop
    while True:
        new_fail_iso: Set[int] = set()
        new_fail_con: Set[int] = set()

        # 1) R_isolated updates
        for node in sorted(R_isolated - failed_R_iso):
            if node in shielded_R:
                continue
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_iso.add(node)

        # 2) R_connected (non-shielded only) updates
        for node in sorted(R_connected_nonshielded - failed_R_con):
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_con.add(node)

        if not new_fail_iso and not new_fail_con:
            break

        for nf in (new_fail_iso | new_fail_con):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con

    return failed_R_iso, failed_R_con

# ---------------------------
# Main simulation
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 2

    surviving_fraction_T = []
    critical_R_fraction = []           # original 2-condition critical R / total R
    critical_R_fraction_coloring = []  # coloring-refined critical R / total R

    for threshold in threshold_values:
        T_frac_runs = []
        critR_frac_runs = []
        critR_color_frac_runs = []

        # per-threshold summary
        Rconn_counts = []
        critR_counts = []
        critR_color_counts = []
        subset_ok_all_runs = True          # original critical ⊆ R_connected (by construction)
        subset_color_ok_all_runs = True    # coloring-critical ⊆ original critical

        for seed in range(n_runs):
            random.seed(seed)
            G = nx.barabasi_albert_graph(nodenumber, 3, seed=seed)

            # Select target nodes as random nodes (mixed degrees)
            top_n = max(1, int(nodenumber * target_pct))
            target_nodes = random.sample(list(G.nodes()), top_n)

            # Initial T/K/R
            T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)

            # TIA refinement
            T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

            # --- C1 split AFTER TIA ---
            R_connected = get_R_connected_direct(G, R_final, K_final)
            R_isolated = R_final - R_connected

            # 1) Original critical R (2 conditions) on current graph
            critical_R = get_critical_shielded_R(G, R_final, R_connected, threshold)

            # 2) Graph-coloring refinement: keep only critical R with degree > lowest avg color degree
            critical_R_coloring = get_coloring_critical_subset(G, critical_R)

            # Collect counts
            Rconn_counts.append(len(R_connected))
            critR_counts.append(len(critical_R))
            critR_color_counts.append(len(critical_R_coloring))
            subset_ok_all_runs &= critical_R.issubset(R_connected)
            subset_color_ok_all_runs &= critical_R_coloring.issubset(critical_R)

            # 3) Cascade: shield ONLY the coloring-based critical R
            _failed_R_iso, _failed_R_con_nonshielded = run_r_isolated_to_r_connected_cascade(
                G_orig=G,
                R_isolated=R_isolated,
                R_connected=R_connected,
                shielded_R=critical_R_coloring,
                threshold=threshold,
                n_initial_failures=n_initial_failures,
            )

            # Metrics

            failed_R_total = _failed_R_iso | _failed_R_con_nonshielded

            failed_K = set()
            failed_T = set()

            init_deg = dict(G.degree())

            # ---- K failure ----
            for k in K_final:
                if any(n in failed_R_total for n in G.neighbors(k)):
                    if init_deg[k] > 0 and (G.degree(k) / init_deg[k]) < threshold:
                       failed_K.add(k)

            # ---- T failure ----
            for t in T_final:
                if any(n in failed_K for n in G.neighbors(t)):
                    if init_deg[t] > 0 and (G.degree(t) / init_deg[t]) < threshold:
                       failed_T.add(t)

            # ---- Survival fraction ----
            survival = 1 - len(failed_T) / max(len(T_final), 1)
            T_frac_runs.append(survival)

            # ---- Critical R fractions ----
            denom = max(len(R_final), 1)
            critR_frac_runs.append(len(critical_R) / denom)
            critR_color_frac_runs.append(len(critical_R_coloring) / denom)

        surviving_fraction_T.append(np.mean(T_frac_runs))
        critical_R_fraction.append(np.mean(critR_frac_runs))
        critical_R_fraction_coloring.append(np.mean(critR_color_frac_runs))

        print(
            f"Threshold={threshold:.2f} | "
            f"Avg Surviving T={np.mean(T_frac_runs):.3f} | "
            f"Avg R_connected(C1)={np.mean(Rconn_counts):.1f} | "
            f"Avg Critical_R(orig)={np.mean(critR_counts):.1f} | "
            f"Avg Critical_R(color)={np.mean(critR_color_counts):.1f} | "
            f"Avg Critical R frac (orig)={np.mean(critR_frac_runs):.3f} | "
            f"Avg Critical R frac (color)={np.mean(critR_color_frac_runs):.3f} | "
            f"subset_ok(orig⊆R_con)={subset_ok_all_runs} | "
            f"subset_ok(color⊆orig)={subset_color_ok_all_runs}"
        )

    # ---------------------------
    # Plot: Surviving T vs Threshold
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, surviving_fraction_T, marker='o', label='Surviving T Fraction')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction")
    plt.title(f"Surviving T Fraction vs Threshold (shielded = coloring-critical; {n_runs} runs)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

    # Combined Plot: Critical R fractions (Original vs Graph Coloring)
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, critical_R_fraction, marker='s', label='Critical R Fraction (Original)')
    plt.plot(threshold_values, critical_R_fraction_coloring, marker='^', label='Critical R Fraction (Graph Coloring)')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction of Total R")
    plt.title("Critical R fraction vs Threshold\nOriginal vs Graph Coloring Based")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()


In [ ]:
# Fig 2 (b) only for the legends : high critical R, high separator K


# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]) -> Tuple[Set[int], Set[int], Set[int]]:
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1: direct neighbors of K ONLY)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Critical shielded R inside R_connected
# ---------------------------
def get_critical_shielded_R(G: nx.Graph, R_final: Set[int], R_connected: Set[int], threshold: float) -> Set[int]:
    init_deg = {n: G.degree(n) for n in R_final}
    fragile_R = {r for r in R_final if init_deg[r] > 0 and (G.degree(r) - 1) / init_deg[r] < threshold}

    critical = set()
    for r in R_connected:
        if r not in fragile_R:
            continue
        fragile_neighbors_in_R = sum(1 for nbr in G.neighbors(r) if (nbr in fragile_R) and (nbr in R_final))
        if fragile_neighbors_in_R >= 2:
            critical.add(r)
    return critical

# ---------------------------
# Graph coloring helper
# ---------------------------
def avg_color_degree(H: nx.Graph):
    colors_dict = nx.coloring.greedy_color(H)
    if not colors_dict:
        return {}, {}
    max_color = max(colors_dict.values())
    avg_color_dict = {}
    for c in range(max_color + 1):
        nodes_c = [n for n, col in colors_dict.items() if col == c]
        if nodes_c:
            degrees = np.array([H.degree(n) for n in nodes_c], dtype=float)
            avg_color_dict[c] = float(np.mean(degrees))
        else:
            avg_color_dict[c] = 0.0
    return colors_dict, avg_color_dict

def get_coloring_critical_subset(G: nx.Graph, critical_R_original: Set[int]) -> Set[int]:
    _, avg_color_dict = avg_color_degree(G)
    if not avg_color_dict:
        return set()
    lowest_avg_deg = min(avg_color_dict.values())
    return {r for r in critical_R_original if G.degree(r) > lowest_avg_deg}

# ---------------------------
# Cascade (shielded set never fails)
# ---------------------------
def run_r_isolated_to_r_connected_cascade(
    G_orig: nx.Graph,
    R_isolated: Set[int],
    R_connected: Set[int],
    shielded_R: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
) -> Tuple[Set[int], Set[int]]:
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}
    R_connected_nonshielded = R_connected - shielded_R

    failed_R_iso: Set[int] = set()
    failed_R_con: Set[int] = set()

    candidates = list(R_isolated)
    if not candidates:
        return failed_R_iso, failed_R_con
    n0 = min(n_initial_failures, len(candidates))
    start_nodes = random.sample(candidates, n0)
    failed_R_iso.update(start_nodes)
    for node in start_nodes:
        if node in G:
            G.remove_node(node)

    while True:
        new_fail_iso: Set[int] = set()
        new_fail_con: Set[int] = set()

        for node in sorted(R_isolated - failed_R_iso):
            if node in shielded_R:
                continue
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_iso.add(node)

        for node in sorted(R_connected_nonshielded - failed_R_con):
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_con.add(node)

        if not new_fail_iso and not new_fail_con:
            break

        for nf in (new_fail_iso | new_fail_con):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con

    return failed_R_iso, failed_R_con

# ---------------------------
# Main simulation (ER version)
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 2

    surviving_fraction_T = []
    critical_R_fraction = []
    critical_R_fraction_coloring = []

    for threshold in threshold_values:
        T_frac_runs = []
        critR_frac_runs = []
        critR_color_frac_runs = []

        Rconn_counts = []
        critR_counts = []
        critR_color_counts = []
        subset_ok_all_runs = True
        subset_color_ok_all_runs = True

        for seed in range(n_runs):
            random.seed(seed)

            # ---- ER GRAPH with avg deg = 6 ----
            p = 6 / nodenumber
            G = nx.erdos_renyi_graph(nodenumber, p, seed=seed)

            top_n = max(1, int(nodenumber * target_pct))
            degree_list = sorted(G.degree, key=lambda x: x[1], reverse=True)
            target_nodes = [node for node, _ in degree_list[:top_n]]

            T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)
            T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

            R_connected = get_R_connected_direct(G, R_final, K_final)
            R_isolated = R_final - R_connected

            critical_R = get_critical_shielded_R(G, R_final, R_connected, threshold)
            critical_R_coloring = get_coloring_critical_subset(G, critical_R)

            Rconn_counts.append(len(R_connected))
            critR_counts.append(len(critical_R))
            critR_color_counts.append(len(critical_R_coloring))

            subset_ok_all_runs &= critical_R.issubset(R_connected)
            subset_color_ok_all_runs &= critical_R_coloring.issubset(critical_R)

            _failed_R_iso, _failed_R_con_nonshielded = run_r_isolated_to_r_connected_cascade(
                G_orig=G,
                R_isolated=R_isolated,
                R_connected=R_connected,
                shielded_R=critical_R_coloring,
                threshold=threshold,
                n_initial_failures=n_initial_failures,
            )
            failed_R_total = _failed_R_iso | _failed_R_con_nonshielded

            failed_K = set()
            failed_T = set()

            init_deg = dict(G.degree())

            # ---- K failure ----
            for k in K_final:
                if any(n in failed_R_total for n in G.neighbors(k)):
                    if init_deg[k] > 0 and (G.degree(k) / init_deg[k]) < threshold:
                       failed_K.add(k)

            # ---- T failure ----
            for t in T_final:
                if any(n in failed_K for n in G.neighbors(t)):
                    if init_deg[t] > 0 and (G.degree(t) / init_deg[t]) < threshold:
                       failed_T.add(t)

            # ---- Survival fraction ----
            survival = 1 - len(failed_T) / max(len(T_final), 1)
            T_frac_runs.append(survival)

            # ---- Critical R fractions ----
            denom = max(len(R_final), 1)
            critR_frac_runs.append(len(critical_R) / denom)
            critR_color_frac_runs.append(len(critical_R_coloring) / denom)

        
            

        surviving_fraction_T.append(np.mean(T_frac_runs))
        critical_R_fraction.append(np.mean(critR_frac_runs))
        critical_R_fraction_coloring.append(np.mean(critR_color_frac_runs))

        print(
            f"Threshold={threshold:.2f} | "
            f"Avg Surviving T={np.mean(T_frac_runs):.3f} | "
            f"Avg R_connected(C1)={np.mean(Rconn_counts):.1f} | "
            f"Avg Critical_R(orig)={np.mean(critR_counts):.1f} | "
            f"Avg Critical_R(color)={np.mean(critR_color_counts):.1f} | "
            f"Avg Critical R frac (orig)={np.mean(critR_frac_runs):.3f} | "
            f"Avg Critical R frac (color)={np.mean(critR_color_frac_runs):.3f} | "
            f"subset_ok(orig⊆R_con)={subset_ok_all_runs} | "
            f"subset_ok(color⊆orig)={subset_color_ok_all_runs}"
        )

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, surviving_fraction_T, marker='o', label='Surviving T Fraction')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction")
    plt.title(f"ER: Surviving T Fraction vs Threshold (shielded = coloring-critical; {n_runs} runs)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, critical_R_fraction, marker='s', label='Critical R Fraction (Original)')
    plt.plot(threshold_values, critical_R_fraction_coloring, marker='^', label='Critical R Fraction (Graph Coloring)')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction of Total R")
    plt.title("ER: Critical R fraction vs Threshold\nOriginal vs Graph Coloring Based")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()


In [ ]:
# Fig 2 (b) only for the legends : low critical R, low separator K


# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]) -> Tuple[Set[int], Set[int], Set[int]]:
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1: direct neighbors of K ONLY)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Critical shielded R inside R_connected
# ---------------------------
def get_critical_shielded_R(G: nx.Graph, R_final: Set[int], R_connected: Set[int], threshold: float) -> Set[int]:
    init_deg = {n: G.degree(n) for n in R_final}
    fragile_R = {r for r in R_final if init_deg[r] > 0 and (G.degree(r) - 1) / init_deg[r] < threshold}

    critical = set()
    for r in R_connected:
        if r not in fragile_R:
            continue
        fragile_neighbors_in_R = sum(1 for nbr in G.neighbors(r) if (nbr in fragile_R) and (nbr in R_final))
        if fragile_neighbors_in_R >= 2:
            critical.add(r)
    return critical

# ---------------------------
# Graph coloring helper
# ---------------------------
def avg_color_degree(H: nx.Graph):
    colors_dict = nx.coloring.greedy_color(H)
    if not colors_dict:
        return {}, {}
    max_color = max(colors_dict.values())
    avg_color_dict = {}
    for c in range(max_color + 1):
        nodes_c = [n for n, col in colors_dict.items() if col == c]
        if nodes_c:
            degrees = np.array([H.degree(n) for n in nodes_c], dtype=float)
            avg_color_dict[c] = float(np.mean(degrees))
        else:
            avg_color_dict[c] = 0.0
    return colors_dict, avg_color_dict

def get_coloring_critical_subset(G: nx.Graph, critical_R_original: Set[int]) -> Set[int]:
    _, avg_color_dict = avg_color_degree(G)
    if not avg_color_dict:
        return set()
    lowest_avg_deg = min(avg_color_dict.values())
    return {r for r in critical_R_original if G.degree(r) > lowest_avg_deg}

# ---------------------------
# Cascade (shielded set never fails)
# ---------------------------
def run_r_isolated_to_r_connected_cascade(
    G_orig: nx.Graph,
    R_isolated: Set[int],
    R_connected: Set[int],
    shielded_R: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
) -> Tuple[Set[int], Set[int]]:
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}
    R_connected_nonshielded = R_connected - shielded_R

    failed_R_iso: Set[int] = set()
    failed_R_con: Set[int] = set()

    candidates = list(R_isolated)
    if not candidates:
        return failed_R_iso, failed_R_con
    n0 = min(n_initial_failures, len(candidates))
    start_nodes = random.sample(candidates, n0)
    failed_R_iso.update(start_nodes)
    for node in start_nodes:
        if node in G:
            G.remove_node(node)

    while True:
        new_fail_iso: Set[int] = set()
        new_fail_con: Set[int] = set()

        for node in sorted(R_isolated - failed_R_iso):
            if node in shielded_R:
                continue
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_iso.add(node)

        for node in sorted(R_connected_nonshielded - failed_R_con):
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_con.add(node)

        if not new_fail_iso and not new_fail_con:
            break

        for nf in (new_fail_iso | new_fail_con):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con

    return failed_R_iso, failed_R_con

# ---------------------------
# Main simulation (ER version)
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 50

    surviving_fraction_T = []
    critical_R_fraction = []
    critical_R_fraction_coloring = []

    for threshold in threshold_values:
        T_frac_runs = []
        critR_frac_runs = []
        critR_color_frac_runs = []

        Rconn_counts = []
        critR_counts = []
        critR_color_counts = []
        subset_ok_all_runs = True
        subset_color_ok_all_runs = True

        for seed in range(n_runs):
            random.seed(seed)

            # ---- ER GRAPH with avg deg = 6 ----
            p = 6 / nodenumber
            G = nx.erdos_renyi_graph(nodenumber, p, seed=seed)

            # Select bottom-degree nodes as target (T)
            top_n = max(1, int(nodenumber * target_pct))
            degree_list = sorted(G.degree, key=lambda x: x[1])  # lowest degree first
            target_nodes = [node for node, _ in degree_list[:top_n]]


            T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)
            T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

            R_connected = get_R_connected_direct(G, R_final, K_final)
            R_isolated = R_final - R_connected

            critical_R = get_critical_shielded_R(G, R_final, R_connected, threshold)
            critical_R_coloring = get_coloring_critical_subset(G, critical_R)

            Rconn_counts.append(len(R_connected))
            critR_counts.append(len(critical_R))
            critR_color_counts.append(len(critical_R_coloring))

            subset_ok_all_runs &= critical_R.issubset(R_connected)
            subset_color_ok_all_runs &= critical_R_coloring.issubset(critical_R)

            _failed_R_iso, _failed_R_con_nonshielded = run_r_isolated_to_r_connected_cascade(
                G_orig=G,
                R_isolated=R_isolated,
                R_connected=R_connected,
                shielded_R=critical_R_coloring,
                threshold=threshold,
                n_initial_failures=n_initial_failures,
            )

            failed_R_total = _failed_R_iso | _failed_R_con_nonshielded

            failed_K = set()
            failed_T = set()

            init_deg = dict(G.degree())

            # ---- K failure ----
            for k in K_final:
                if any(n in failed_R_total for n in G.neighbors(k)):
                    if init_deg[k] > 0 and (G.degree(k) / init_deg[k]) < threshold:
                       failed_K.add(k)

            # ---- T failure ----
            for t in T_final:
                if any(n in failed_K for n in G.neighbors(t)):
                    if init_deg[t] > 0 and (G.degree(t) / init_deg[t]) < threshold:
                       failed_T.add(t)

            # ---- Survival fraction ----
            survival = 1 - len(failed_T) / max(len(T_final), 1)
            T_frac_runs.append(survival)

            # ---- Critical R fractions ----
            denom = max(len(R_final), 1)
            critR_frac_runs.append(len(critical_R) / denom)
            critR_color_frac_runs.append(len(critical_R_coloring) / denom)

       

        surviving_fraction_T.append(np.mean(T_frac_runs))
        critical_R_fraction.append(np.mean(critR_frac_runs))
        critical_R_fraction_coloring.append(np.mean(critR_color_frac_runs))

        print(
            f"Threshold={threshold:.2f} | "
            f"Avg Surviving T={np.mean(T_frac_runs):.3f} | "
            f"Avg R_connected(C1)={np.mean(Rconn_counts):.1f} | "
            f"Avg Critical_R(orig)={np.mean(critR_counts):.1f} | "
            f"Avg Critical_R(color)={np.mean(critR_color_counts):.1f} | "
            f"Avg Critical R frac (orig)={np.mean(critR_frac_runs):.3f} | "
            f"Avg Critical R frac (color)={np.mean(critR_color_frac_runs):.3f} | "
            f"subset_ok(orig⊆R_con)={subset_ok_all_runs} | "
            f"subset_ok(color⊆orig)={subset_color_ok_all_runs}"
        )

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, surviving_fraction_T, marker='o', label='Surviving T Fraction')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction")
    plt.title(f"ER: Surviving T Fraction vs Threshold (shielded = coloring-critical; {n_runs} runs)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, critical_R_fraction, marker='s', label='Critical R Fraction (Original)')
    plt.plot(threshold_values, critical_R_fraction_coloring, marker='^', label='Critical R Fraction (Graph Coloring)')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction of Total R")
    plt.title("ER: Critical R fraction vs Threshold\nOriginal vs Graph Coloring Based")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()


In [ ]:
# Fig 2 (b) only for the legends : random critical R, random separator K


# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]) -> Tuple[Set[int], Set[int], Set[int]]:
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1: direct neighbors of K ONLY)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Critical shielded R inside R_connected
# ---------------------------
def get_critical_shielded_R(G: nx.Graph, R_final: Set[int], R_connected: Set[int], threshold: float) -> Set[int]:
    init_deg = {n: G.degree(n) for n in R_final}
    fragile_R = {r for r in R_final if init_deg[r] > 0 and (G.degree(r) - 1) / init_deg[r] < threshold}

    critical = set()
    for r in R_connected:
        if r not in fragile_R:
            continue
        fragile_neighbors_in_R = sum(1 for nbr in G.neighbors(r) if (nbr in fragile_R) and (nbr in R_final))
        if fragile_neighbors_in_R >= 2:
            critical.add(r)
    return critical

# ---------------------------
# Graph coloring helper
# ---------------------------
def avg_color_degree(H: nx.Graph):
    colors_dict = nx.coloring.greedy_color(H)
    if not colors_dict:
        return {}, {}
    max_color = max(colors_dict.values())
    avg_color_dict = {}
    for c in range(max_color + 1):
        nodes_c = [n for n, col in colors_dict.items() if col == c]
        if nodes_c:
            degrees = np.array([H.degree(n) for n in nodes_c], dtype=float)
            avg_color_dict[c] = float(np.mean(degrees))
        else:
            avg_color_dict[c] = 0.0
    return colors_dict, avg_color_dict

def get_coloring_critical_subset(G: nx.Graph, critical_R_original: Set[int]) -> Set[int]:
    _, avg_color_dict = avg_color_degree(G)
    if not avg_color_dict:
        return set()
    lowest_avg_deg = min(avg_color_dict.values())
    return {r for r in critical_R_original if G.degree(r) > lowest_avg_deg}

# ---------------------------
# Cascade (shielded set never fails)
# ---------------------------
def run_r_isolated_to_r_connected_cascade(
    G_orig: nx.Graph,
    R_isolated: Set[int],
    R_connected: Set[int],
    shielded_R: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
) -> Tuple[Set[int], Set[int]]:
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}
    R_connected_nonshielded = R_connected - shielded_R

    failed_R_iso: Set[int] = set()
    failed_R_con: Set[int] = set()

    candidates = list(R_isolated)
    if not candidates:
        return failed_R_iso, failed_R_con
    n0 = min(n_initial_failures, len(candidates))
    start_nodes = random.sample(candidates, n0)
    failed_R_iso.update(start_nodes)
    for node in start_nodes:
        if node in G:
            G.remove_node(node)

    while True:
        new_fail_iso: Set[int] = set()
        new_fail_con: Set[int] = set()

        for node in sorted(R_isolated - failed_R_iso):
            if node in shielded_R:
                continue
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_iso.add(node)

        for node in sorted(R_connected_nonshielded - failed_R_con):
            neighs = list(G_orig.neighbors(node))
            if any(n in failed_R_iso or n in failed_R_con for n in neighs):
                cur = G.degree(node) if node in G else 0
                ini = init_deg_all[node]
                if ini > 0 and (cur / ini) < threshold:
                    new_fail_con.add(node)

        if not new_fail_iso and not new_fail_con:
            break

        for nf in (new_fail_iso | new_fail_con):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con

    return failed_R_iso, failed_R_con

# ---------------------------
# Main simulation (ER version)
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 50

    surviving_fraction_T = []
    critical_R_fraction = []
    critical_R_fraction_coloring = []

    for threshold in threshold_values:
        T_frac_runs = []
        critR_frac_runs = []
        critR_color_frac_runs = []

        Rconn_counts = []
        critR_counts = []
        critR_color_counts = []
        subset_ok_all_runs = True
        subset_color_ok_all_runs = True

        for seed in range(n_runs):
            random.seed(seed)

            # ---- ER GRAPH with avg deg = 6 ----
            p = 6 / nodenumber
            G = nx.erdos_renyi_graph(nodenumber, p, seed=seed)

            # Select target nodes as random nodes (mixed degrees)
            top_n = max(1, int(nodenumber * target_pct))
            target_nodes = random.sample(list(G.nodes()), top_n)

            T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)
            T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

            R_connected = get_R_connected_direct(G, R_final, K_final)
            R_isolated = R_final - R_connected

            critical_R = get_critical_shielded_R(G, R_final, R_connected, threshold)
            critical_R_coloring = get_coloring_critical_subset(G, critical_R)

            Rconn_counts.append(len(R_connected))
            critR_counts.append(len(critical_R))
            critR_color_counts.append(len(critical_R_coloring))

            subset_ok_all_runs &= critical_R.issubset(R_connected)
            subset_color_ok_all_runs &= critical_R_coloring.issubset(critical_R)

            _failed_R_iso, _failed_R_con_nonshielded = run_r_isolated_to_r_connected_cascade(
                G_orig=G,
                R_isolated=R_isolated,
                R_connected=R_connected,
                shielded_R=critical_R_coloring,
                threshold=threshold,
                n_initial_failures=n_initial_failures,
            )

            failed_R_total = _failed_R_iso | _failed_R_con_nonshielded

            failed_K = set()
            failed_T = set()

            init_deg = dict(G.degree())

            # ---- K failure ----
            for k in K_final:
                if any(n in failed_R_total for n in G.neighbors(k)):
                    if init_deg[k] > 0 and (G.degree(k) / init_deg[k]) < threshold:
                       failed_K.add(k)

            # ---- T failure ----
            for t in T_final:
                if any(n in failed_K for n in G.neighbors(t)):
                    if init_deg[t] > 0 and (G.degree(t) / init_deg[t]) < threshold:
                       failed_T.add(t)

            # ---- Survival fraction ----
            survival = 1 - len(failed_T) / max(len(T_final), 1)
            T_frac_runs.append(survival)

            # ---- Critical R fractions ----
            denom = max(len(R_final), 1)
            critR_frac_runs.append(len(critical_R) / denom)
            critR_color_frac_runs.append(len(critical_R_coloring) / denom)

        

        surviving_fraction_T.append(np.mean(T_frac_runs))
        critical_R_fraction.append(np.mean(critR_frac_runs))
        critical_R_fraction_coloring.append(np.mean(critR_color_frac_runs))

        print(
            f"Threshold={threshold:.2f} | "
            f"Avg Surviving T={np.mean(T_frac_runs):.3f} | "
            f"Avg R_connected(C1)={np.mean(Rconn_counts):.1f} | "
            f"Avg Critical_R(orig)={np.mean(critR_counts):.1f} | "
            f"Avg Critical_R(color)={np.mean(critR_color_counts):.1f} | "
            f"Avg Critical R frac (orig)={np.mean(critR_frac_runs):.3f} | "
            f"Avg Critical R frac (color)={np.mean(critR_color_frac_runs):.3f} | "
            f"subset_ok(orig⊆R_con)={subset_ok_all_runs} | "
            f"subset_ok(color⊆orig)={subset_color_ok_all_runs}"
        )

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, surviving_fraction_T, marker='o', label='Surviving T Fraction')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction")
    plt.title(f"ER: Surviving T Fraction vs Threshold (shielded = coloring-critical; {n_runs} runs)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, critical_R_fraction, marker='s', label='Critical R Fraction (Original)')
    plt.plot(threshold_values, critical_R_fraction_coloring, marker='^', label='Critical R Fraction (Graph Coloring)')
    plt.xlabel("Threshold")
    plt.ylabel("Fraction of Total R")
    plt.title("ER: Critical R fraction vs Threshold\nOriginal vs Graph Coloring Based")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()


In [ ]:
# Fig 2 (a) and (b) for the legend: no shield high

# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]):
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Unshielded cascade across all sets
# ---------------------------
def run_unshielded_cascade_all_sets(
    G_orig: nx.Graph,
    T_final: Set[int],
    K_final: Set[int],
    R_isolated: Set[int],
    R_connected: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
):
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}

    failed_R_iso, failed_R_con, failed_K, failed_T = set(), set(), set(), set()

    # Seed failures in R_isolated only
    candidates = list(R_isolated)
    if candidates:
        start_nodes = random.sample(candidates, min(n_initial_failures, len(candidates)))
        failed_R_iso.update(start_nodes)
        for node in start_nodes:
            if node in G:
                G.remove_node(node)

    def step_fail(nodes: Set[int]) -> Set[int]:
        new_fail = set()
        for node in sorted(nodes):
            if not any(n in failed_R_iso or n in failed_R_con or n in failed_K or n in failed_T for n in G_orig.neighbors(node)):
                continue
            cur = G.degree(node) if node in G else 0
            ini = init_deg_all[node]
            if ini > 0 and (cur / ini) < threshold:
                new_fail.add(node)
        return new_fail

    while True:
        new_fail_iso = step_fail(R_isolated - failed_R_iso)
        new_fail_con = step_fail(R_connected - failed_R_con)
        new_fail_K = step_fail(K_final - failed_K)
        new_fail_T = step_fail(T_final - failed_T)

        if not (new_fail_iso or new_fail_con or new_fail_K or new_fail_T):
            break

        for nf in (new_fail_iso | new_fail_con | new_fail_K | new_fail_T):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con
        failed_K |= new_fail_K
        failed_T |= new_fail_T

    return failed_T


# ---------------------------
# Main: run BA & ER models
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 2

    results = {"BA": [], "ER": []}

    for model in ["BA", "ER"]:
        for threshold in threshold_values:
            T_frac_runs = []
            for seed in range(n_runs):
                random.seed(seed)

                # Graph generation
                if model == "BA":
                    G = nx.barabasi_albert_graph(nodenumber, 3, seed=seed)
                else:
                    p = 6 / nodenumber
                    G = nx.erdos_renyi_graph(nodenumber, p, seed=seed)

                # Select random nodes as T
                top_n = max(1, int(nodenumber * target_pct))
                target_nodes = random.sample(list(G.nodes()), top_n)

                # Initial T/K/R
                T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)

                # TIA refinement
                T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

                # Split R
                R_connected = get_R_connected_direct(G, R_final, K_final)
                R_isolated = R_final - R_connected

                # Run cascade
                failed_T = run_unshielded_cascade_all_sets(
                    G_orig=G,
                    T_final=T_final,
                    K_final=K_final,
                    R_isolated=R_isolated,
                    R_connected=R_connected,
                    threshold=threshold,
                    n_initial_failures=n_initial_failures,
                )

                t_denom = max(len(T_final), 1)
                T_frac_runs.append((t_denom - len(failed_T)) / t_denom)

            results[model].append(np.mean(T_frac_runs))

    # ---------------------------
    # Print only surviving T fraction
    # ---------------------------
    print("\n=== Surviving T Fraction for BA and ER ===")
    for thr, ba_val, er_val in zip(threshold_values, results["BA"], results["ER"]):
        print(f"Threshold={thr:.2f} | BA={ba_val:.3f} | ER={er_val:.3f}")

    # ---------------------------
    # Plot
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, results["BA"], marker='o', label='BA (m=3)')
    plt.plot(threshold_values, results["ER"], marker='s', label='ER (avg deg~6)')
    plt.xlabel("Threshold")
    plt.ylabel("Surviving T Fraction")
    plt.title("Surviving T Fraction vs Threshold (Unshielded R→K→T)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()


In [ ]:
# Fig 2 (a) and (b) for the legend: no shield low
import networkx as nx
import random
from typing import Set, Tuple, List
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]):
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Unshielded cascade across all sets
# ---------------------------
def run_unshielded_cascade_all_sets(
    G_orig: nx.Graph,
    T_final: Set[int],
    K_final: Set[int],
    R_isolated: Set[int],
    R_connected: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
):
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}

    failed_R_iso, failed_R_con, failed_K, failed_T = set(), set(), set(), set()

    # Seed failures in R_isolated only
    candidates = list(R_isolated)
    if candidates:
        start_nodes = random.sample(candidates, min(n_initial_failures, len(candidates)))
        failed_R_iso.update(start_nodes)
        for node in start_nodes:
            if node in G:
                G.remove_node(node)

    def step_fail(nodes: Set[int]) -> Set[int]:
        new_fail = set()
        for node in sorted(nodes):
            if not any(n in failed_R_iso or n in failed_R_con or n in failed_K or n in failed_T for n in G_orig.neighbors(node)):
                continue
            cur = G.degree(node) if node in G else 0
            ini = init_deg_all[node]
            if ini > 0 and (cur / ini) < threshold:
                new_fail.add(node)
        return new_fail

    while True:
        new_fail_iso = step_fail(R_isolated - failed_R_iso)
        new_fail_con = step_fail(R_connected - failed_R_con)
        new_fail_K = step_fail(K_final - failed_K)
        new_fail_T = step_fail(T_final - failed_T)

        if not (new_fail_iso or new_fail_con or new_fail_K or new_fail_T):
            break

        for nf in (new_fail_iso | new_fail_con | new_fail_K | new_fail_T):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con
        failed_K |= new_fail_K
        failed_T |= new_fail_T

    return failed_T


# ---------------------------
# Main: run BA & ER models
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 2

    results = {"BA": [], "ER": []}

    for model in ["BA", "ER"]:
        for threshold in threshold_values:
            T_frac_runs = []
            for seed in range(n_runs):
                random.seed(seed)

                # Graph generation
                if model == "BA":
                    G = nx.barabasi_albert_graph(nodenumber, 3, seed=seed)
                else:
                    p = 6 / nodenumber
                    G = nx.erdos_renyi_graph(nodenumber, p, seed=seed)

                # Select bottom-degree nodes as target (T)
                top_n = max(1, int(nodenumber * target_pct))
                degree_list = sorted(G.degree, key=lambda x: x[1])  # lowest degree first
                target_nodes = [node for node, _ in degree_list[:top_n]]

                # Initial T/K/R
                T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)

                # TIA refinement
                T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

                # Split R
                R_connected = get_R_connected_direct(G, R_final, K_final)
                R_isolated = R_final - R_connected

                # Run cascade
                failed_T = run_unshielded_cascade_all_sets(
                    G_orig=G,
                    T_final=T_final,
                    K_final=K_final,
                    R_isolated=R_isolated,
                    R_connected=R_connected,
                    threshold=threshold,
                    n_initial_failures=n_initial_failures,
                )

                t_denom = max(len(T_final), 1)
                T_frac_runs.append((t_denom - len(failed_T)) / t_denom)

            results[model].append(np.mean(T_frac_runs))

    # ---------------------------
    # Print only surviving T fraction
    # ---------------------------
    print("\n=== Surviving T Fraction for BA and ER ===")
    for thr, ba_val, er_val in zip(threshold_values, results["BA"], results["ER"]):
        print(f"Threshold={thr:.2f} | BA={ba_val:.3f} | ER={er_val:.3f}")

    # ---------------------------
    # Plot
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, results["BA"], marker='o', label='BA (m=3)')
    plt.plot(threshold_values, results["ER"], marker='s', label='ER (avg deg~6)')
    plt.xlabel("Threshold")
    plt.ylabel("Surviving T Fraction")
    plt.title("Surviving T Fraction vs Threshold (Unshielded R→K→T)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()


In [ ]:
# Fig 2 (a) and (b) for the legend: no shield random
import networkx as nx
import random
from typing import Set, Tuple, List
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# Helper: Initial T/K/R
# ---------------------------
def manual_initial_TKR_no_threshold(G: nx.Graph, target_nodes: List[int]):
    T = set(target_nodes)
    K = set()
    for t in T:
        K.update(G.neighbors(t))
    K -= T
    R = set(G.nodes) - T - K
    return T, K, R

# ---------------------------
# TIA Algorithm
# ---------------------------
def tia_large_graph(G: nx.Graph, T: Set[int], K: Set[int], R: Set[int], max_iter: int = 100):
    iteration = 0
    while iteration < max_iter:
        iteration += 1
        prev_T, prev_K, prev_R = set(T), set(K), set(R)

        # K -> T if K has no neighbors in R
        move_to_T = {node for node in K if not any(neigh in R for neigh in G.neighbors(node))}
        T.update(move_to_T)
        K -= move_to_T

        # R -> K if R has any neighbor in T
        move_to_K = {node for node in R if any(neigh in T for neigh in G.neighbors(node))}
        K.update(move_to_K)
        R -= move_to_K

        if (T, K, R) == (prev_T, prev_K, prev_R):
            break
    return T, K, R

# ---------------------------
# R_connected (C1)
# ---------------------------
def get_R_connected_direct(G: nx.Graph, R_final: Set[int], K_final: Set[int]) -> Set[int]:
    return {r for r in R_final if any(neigh in K_final for neigh in G.neighbors(r))}

# ---------------------------
# Unshielded cascade across all sets
# ---------------------------
def run_unshielded_cascade_all_sets(
    G_orig: nx.Graph,
    T_final: Set[int],
    K_final: Set[int],
    R_isolated: Set[int],
    R_connected: Set[int],
    threshold: float = 0.95,
    n_initial_failures: int = 5,
):
    G = G_orig.copy()
    init_deg_all = {n: G_orig.degree(n) for n in G_orig.nodes()}

    failed_R_iso, failed_R_con, failed_K, failed_T = set(), set(), set(), set()

    # Seed failures in R_isolated only
    candidates = list(R_isolated)
    if candidates:
        start_nodes = random.sample(candidates, min(n_initial_failures, len(candidates)))
        failed_R_iso.update(start_nodes)
        for node in start_nodes:
            if node in G:
                G.remove_node(node)

    def step_fail(nodes: Set[int]) -> Set[int]:
        new_fail = set()
        for node in sorted(nodes):
            if not any(n in failed_R_iso or n in failed_R_con or n in failed_K or n in failed_T for n in G_orig.neighbors(node)):
                continue
            cur = G.degree(node) if node in G else 0
            ini = init_deg_all[node]
            if ini > 0 and (cur / ini) < threshold:
                new_fail.add(node)
        return new_fail

    while True:
        new_fail_iso = step_fail(R_isolated - failed_R_iso)
        new_fail_con = step_fail(R_connected - failed_R_con)
        new_fail_K = step_fail(K_final - failed_K)
        new_fail_T = step_fail(T_final - failed_T)

        if not (new_fail_iso or new_fail_con or new_fail_K or new_fail_T):
            break

        for nf in (new_fail_iso | new_fail_con | new_fail_K | new_fail_T):
            if nf in G:
                G.remove_node(nf)

        failed_R_iso |= new_fail_iso
        failed_R_con |= new_fail_con
        failed_K |= new_fail_K
        failed_T |= new_fail_T

    return failed_T


# ---------------------------
# Main: run BA & ER models
# ---------------------------
if __name__ == "__main__":
    nodenumber = 10000
    target_pct = 0.15
    n_runs = 10
    threshold_values = np.arange(0.80, 0.98, 0.01)
    n_initial_failures = 2

    results = {"BA": [], "ER": []}

    for model in ["BA", "ER"]:
        for threshold in threshold_values:
            T_frac_runs = []
            for seed in range(n_runs):
                random.seed(seed)

                # Graph generation
                if model == "BA":
                    G = nx.barabasi_albert_graph(nodenumber, 3, seed=seed)
                else:
                    p = 6 / nodenumber
                    G = nx.erdos_renyi_graph(nodenumber, p, seed=seed)

                # Select target nodes as random nodes (mixed degrees)
                top_n = max(1, int(nodenumber * target_pct))
                target_nodes = random.sample(list(G.nodes()), top_n) 

                # Initial T/K/R
                T, K, R = manual_initial_TKR_no_threshold(G, target_nodes)

                # TIA refinement
                T_final, K_final, R_final = tia_large_graph(G.copy(), T.copy(), K.copy(), R.copy())

                # Split R
                R_connected = get_R_connected_direct(G, R_final, K_final)
                R_isolated = R_final - R_connected

                # Run cascade
                failed_T = run_unshielded_cascade_all_sets(
                    G_orig=G,
                    T_final=T_final,
                    K_final=K_final,
                    R_isolated=R_isolated,
                    R_connected=R_connected,
                    threshold=threshold,
                    n_initial_failures=n_initial_failures,
                )

                t_denom = max(len(T_final), 1)
                T_frac_runs.append((t_denom - len(failed_T)) / t_denom)

            results[model].append(np.mean(T_frac_runs))

    # ---------------------------
    # Print only surviving T fraction
    # ---------------------------
    print("\n=== Surviving T Fraction for BA and ER ===")
    for thr, ba_val, er_val in zip(threshold_values, results["BA"], results["ER"]):
        print(f"Threshold={thr:.2f} | BA={ba_val:.3f} | ER={er_val:.3f}")

    # ---------------------------
    # Plot
    # ---------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(threshold_values, results["BA"], marker='o', label='BA (m=3)')
    plt.plot(threshold_values, results["ER"], marker='s', label='ER (avg deg~6)')
    plt.xlabel("Threshold")
    plt.ylabel("Surviving T Fraction")
    plt.title("Surviving T Fraction vs Threshold (Unshielded R→K→T)")
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()
